# Notebook 2 — Aggregation & Database Schema

**Team 8 — Delhi Air Quality Pipeline**
**Member: [Your Name Here]**

## What this notebook does
This is the second stage of the pipeline. My job is to:
1. Read the partitioned data that Member 1 produced
2. Aggregate the 15-minute readings into hourly averages
3. Design a normalized SQLite schema (3NF) with three tables
4. Load the aggregated data into SQLite

The output of this notebook (`delhi_air_quality.db`) is what Member 3
uses to run their SQL analytics queries.

## Why aggregation matters
The raw data has readings every 15 minutes — that's 22.5 million rows. For
analytical queries, hourly averages give the same insights with 60× fewer
rows (~250K). This is a standard ETL pattern: aggregate before loading into
the warehouse so queries stay fast.


## Step 1: Setup (same as notebook 1)

In [ ]:
!apt-get install -y openjdk-17-jdk-headless -qq > /dev/null
!pip install pyspark -q

import os
os.environ["JAVA_HOME"] = "/usr/lib/jvm/java-17-openjdk-amd64"
print("Java ready")

## Step 2: Get the data from Member 1

Two options here — either upload the partitioned data, or just use the
original parquet file. The original file is simpler for re-running, so
we'll go with that.

In [ ]:
from google.colab import files
uploaded = files.upload()

In [ ]:
from pyspark.sql import SparkSession
from pyspark.sql import functions as F

spark = (
    SparkSession.builder
    .appName("Team8_Aggregation")
    .config("spark.driver.memory", "4g")
    .getOrCreate()
)

df = spark.read.parquet("team_8.parquet")
print(f"Loaded {df.count():,} rows")

## Step 3: Aggregate to hourly averages

This is the main work of stage 2. We group by (station, pollutant, year, month,
day, hour) and compute averages of all the numeric columns.

The result: every (station, pollutant, hour) combination becomes one row with
its average, min, max pollutant value plus the average weather conditions
during that hour.

In [ ]:
hourly = (
    df.groupBy(
        "station_id", "station_name",
        "year", "month", "day", "hour",
        "pollutant"
    )
    .agg(
        F.avg("value").alias("avg_value"),
        F.min("value").alias("min_value"),
        F.max("value").alias("max_value"),
        F.count("value").alias("reading_count"),
        F.avg("at_c").alias("avg_temp_c"),
        F.avg("rh_percent").alias("avg_humidity"),
        F.avg("ws_m_s").alias("avg_wind_speed"),
        F.avg("rf_mm").alias("avg_rainfall"),
    )
)

print(f"Aggregated row count: {hourly.count():,}")
hourly.show(5)

## Step 4: Design the database schema

I'm using **third normal form (3NF)** — three tables connected by foreign keys,
no repeated data:

**`stations`** — one row per monitoring station
| Column | Type | Notes |
|---|---|---|
| station_id | TEXT | PRIMARY KEY |
| station_name | TEXT | NOT NULL |

**`pollutants`** — one row per pollutant type
| Column | Type | Notes |
|---|---|---|
| pollutant_id | INTEGER | PRIMARY KEY, AUTOINCREMENT |
| pollutant_name | TEXT | UNIQUE, NOT NULL |

**`measurements`** — the big fact table, one row per station/pollutant/hour
| Column | Type | Notes |
|---|---|---|
| id | INTEGER | PRIMARY KEY |
| station_id | TEXT | FOREIGN KEY → stations |
| pollutant_id | INTEGER | FOREIGN KEY → pollutants |
| year, month, day, hour | INTEGER | the time slot |
| avg_value, min_value, max_value | REAL | pollutant statistics |
| avg_temp_c, avg_humidity, avg_wind_speed, avg_rainfall | REAL | weather context |

Why this works: instead of repeating "Alipur, Delhi - DPCC" for thousands of
rows, we store it once in `stations` and reference it by ID in `measurements`.
Same for pollutants. This is what normalization means.

In [ ]:
import sqlite3

conn = sqlite3.connect("delhi_air_quality.db")
cur = conn.cursor()

# clear out anything that already exists (so the cell can re-run)
cur.execute("DROP TABLE IF EXISTS measurements")
cur.execute("DROP TABLE IF EXISTS stations")
cur.execute("DROP TABLE IF EXISTS pollutants")

# stations table
cur.execute('''
    CREATE TABLE stations (
        station_id TEXT PRIMARY KEY,
        station_name TEXT NOT NULL
    )
''')

# pollutants table
cur.execute('''
    CREATE TABLE pollutants (
        pollutant_id INTEGER PRIMARY KEY AUTOINCREMENT,
        pollutant_name TEXT NOT NULL UNIQUE
    )
''')

# measurements table (the fact table)
cur.execute('''
    CREATE TABLE measurements (
        id INTEGER PRIMARY KEY AUTOINCREMENT,
        station_id TEXT NOT NULL,
        pollutant_id INTEGER NOT NULL,
        year INTEGER NOT NULL,
        month INTEGER NOT NULL,
        day INTEGER NOT NULL,
        hour INTEGER NOT NULL,
        avg_value REAL,
        min_value REAL,
        max_value REAL,
        avg_temp_c REAL,
        avg_humidity REAL,
        avg_wind_speed REAL,
        avg_rainfall REAL,
        FOREIGN KEY (station_id) REFERENCES stations(station_id),
        FOREIGN KEY (pollutant_id) REFERENCES pollutants(pollutant_id)
    )
''')

# add indexes so the analytics queries (Member 3's work) run fast
cur.execute("CREATE INDEX idx_meas_station ON measurements(station_id)")
cur.execute("CREATE INDEX idx_meas_pollutant ON measurements(pollutant_id)")
cur.execute("CREATE INDEX idx_meas_date ON measurements(year, month)")

conn.commit()
print("All three tables created")

## Step 5: Load the data into SQLite

In [ ]:
import pandas as pd

# stations table — one row per station
stations_pd = df.select("station_id", "station_name").distinct().toPandas()
stations_pd.to_sql("stations", conn, if_exists="append", index=False)
print(f"Inserted {len(stations_pd)} stations")
print(stations_pd.head())

In [ ]:
# pollutants table — one row per pollutant
pollutants_pd = df.select("pollutant").distinct().toPandas()
pollutants_pd = pollutants_pd.rename(columns={"pollutant": "pollutant_name"})
pollutants_pd.to_sql("pollutants", conn, if_exists="append", index=False)
print(f"Inserted {len(pollutants_pd)} pollutants")

# show the pollutants with their auto-assigned IDs
print(pd.read_sql("SELECT * FROM pollutants ORDER BY pollutant_id", conn))

In [ ]:
# now the big one — the measurements table
# pull the aggregated hourly data from Spark into pandas
hourly_pd = hourly.toPandas()
print(f"Got {len(hourly_pd):,} hourly aggregates from Spark")

# replace the pollutant name with its numeric ID (so we can use it as a foreign key)
pol_lookup = dict(pd.read_sql(
    "SELECT pollutant_name, pollutant_id FROM pollutants", conn
).values)
hourly_pd["pollutant_id"] = hourly_pd["pollutant"].map(pol_lookup)

# drop the columns we don't need in the measurements table
to_insert = hourly_pd.drop(columns=["pollutant", "station_name"])

# insert in chunks of 10k to keep memory steady
to_insert.to_sql("measurements", conn, if_exists="append", index=False, chunksize=10000)
conn.commit()
print(f"Inserted {len(to_insert):,} measurements")

## Step 6: Verify the database

In [ ]:
# row counts in each table
print("Stations     :", pd.read_sql("SELECT COUNT(*) AS n FROM stations", conn).iloc[0, 0])
print("Pollutants   :", pd.read_sql("SELECT COUNT(*) AS n FROM pollutants", conn).iloc[0, 0])
print("Measurements :", pd.read_sql("SELECT COUNT(*) AS n FROM measurements", conn).iloc[0, 0])

In [ ]:
# quick test query — show a few joined rows to prove the foreign keys work
test_query = '''
SELECT
    s.station_name,
    p.pollutant_name,
    m.year, m.month, m.day, m.hour,
    ROUND(m.avg_value, 2) AS avg_value
FROM measurements m
JOIN stations s ON m.station_id = s.station_id
JOIN pollutants p ON m.pollutant_id = p.pollutant_id
LIMIT 5
'''
pd.read_sql(test_query, conn)

In [ ]:
conn.close()
print("Database is ready to hand off to Member 3")

## Step 7: Download the database

Send the `.db` file to Member 3 for the analytics stage.

In [ ]:
files.download("delhi_air_quality.db")

## What I did (for the class presentation)

> "I handled the data warehouse layer. I used PySpark to aggregate the raw
> 15-minute readings into hourly averages — going from 22 million rows to
> about 250 thousand without losing analytical value. Then I designed a
> normalized SQLite schema in third normal form: three tables (stations,
> pollutants, measurements) connected by foreign keys, with indexes on the
> columns we query most often. Aggregating before loading into the warehouse
> is a standard ETL pattern that keeps queries fast."

## Key concepts to know
- **Aggregation** — grouping rows and computing a summary (avg, min, max, count) for each group
- **Third Normal Form (3NF)** — schema design where no column depends on a non-key column; no repeated data
- **Foreign keys** — references from one table's rows to another's, enforcing data integrity
- **Indexes** — extra data structures that make `WHERE` and `JOIN` queries much faster
- **ETL** — Extract, Transform, Load — the standard pattern of taking raw data, processing it, and loading into a warehouse
